# Explainable AI Email Triage: Week 6 Testing and Debugging

This experiment log evaluates the frozen Week 4 email-triage candidate without changing the submitted Week 3 or Week 4 notebooks. Week 6 uses cross-validation because this project has labeled historical data but no deployed system, live traffic, or safe treatment-and-control environment for an A/B test.

**Checkpoint 1 status:** This version establishes the notebook structure, freezes the model and testing parameters, validates the reviewed modeling population, and recreates the preserved training and preliminary holdout split. It does not run threshold selection or report Week 6 performance results.

## tl;dr

Checkpoint 1 confirms the required local inputs and reproduces the approved modeling population of 199 reviewed messages: 169 nonurgent and 30 urgent. The fixed stratified split contains 159 training messages and 40 preliminary holdout messages. The holdout remains sealed for model and threshold selection. No raw email text appears in notebook outputs or exported evidence.

## Context and methods

The Week 6 assignment requires updated code with test results and a 2–3-page debugging and evaluation report. The testing workflow will use the frozen Week 4 candidate, nested stratified cross-validation, a preregistered probability-threshold grid, privacy-safe error analysis, and one preliminary holdout comparison after every training-data decision is locked.

The inner folds will select the probability threshold. The outer folds will evaluate that selection process on observations the inner loop did not use. This separation follows scikit-learn's guidance on [nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html). The notebook will emphasize urgent recall, urgent F1, urgent false negatives, and false-positive review burden rather than accuracy alone.

### Key assumptions and boundaries

- The reviewed label manifest is the label source of truth.
- The Week 4 candidate remains frozen; Week 6 will not reopen broad hyperparameter tuning.
- All threshold selection will remain inside the 159-message training partition.
- The 40-message holdout is preliminary comparison evidence because earlier project work already used it.
- The model remains decision support for a person. No result in this notebook approves autonomous email handling.
- Privacy-safe outputs may contain identifiers, labels, probabilities, error types, approved categories, and aggregate metrics. They must not contain subjects, bodies, excerpts, reviewer notes, or unnecessary personal information.

## Setup and reproducibility

The setup cells record the environment and freeze every parameter needed for the later testing workflow before loading the data.

In [1]:
# I record the analysis environment before loading project data.
import gc
import hashlib
import re
import sys
from email import policy
from email.parser import Parser
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.11.8
NumPy: 2.4.6
pandas: 3.0.3
scikit-learn: 1.9.0


In [2]:
# I freeze the Week 6 configuration before running any model evaluation.
SEED = 42
TEST_SIZE = 0.20
OUTER_FOLDS = 5
INNER_FOLDS = 4
THRESHOLDS = (0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50)
LABEL_ORDER = ("nonurgent", "urgent")

EXPECTED_RECORDS = 199
EXPECTED_LABEL_COUNTS = {"nonurgent": 169, "urgent": 30}
EXPECTED_TRAIN_RECORDS = 159
EXPECTED_HOLDOUT_RECORDS = 40
EXPECTED_TRAIN_LABEL_COUNTS = {"nonurgent": 135, "urgent": 24}
EXPECTED_HOLDOUT_LABEL_COUNTS = {"nonurgent": 34, "urgent": 6}

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / "data" / "raw" / "emails.csv"
LABEL_MANIFEST = PROJECT_ROOT / "data" / "labels" / "enron_urgency_labels_v1.csv"

missing_inputs = [path for path in (RAW_DATA, LABEL_MANIFEST) if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Missing required input(s): " + ", ".join(str(path) for path in missing_inputs)
    )

label_manifest_sha256 = hashlib.sha256(LABEL_MANIFEST.read_bytes()).hexdigest()
input_inventory = pd.DataFrame(
    [
        {
            "input": "Raw Enron source",
            "path": str(RAW_DATA.relative_to(PROJECT_ROOT)),
            "bytes": RAW_DATA.stat().st_size,
            "sha256": "Not calculated for the 1.4 GB local source",
        },
        {
            "input": "Reviewed label manifest",
            "path": str(LABEL_MANIFEST.relative_to(PROJECT_ROOT)),
            "bytes": LABEL_MANIFEST.stat().st_size,
            "sha256": label_manifest_sha256,
        },
    ]
)
display(input_inventory)

,input,path,bytes,sha256
0,Raw Enron source,data/raw/emails.csv,1426122219,Not calculated for the 1.4 GB local source
1,Reviewed label manifest,data/labels/enron_urgency_labels_v1.csv,13038,1d97ae4369cfe86cd53e34f97e5d3ebc74c6f03e5fb485...


### Freeze the Week 4 candidate

The candidate combines unigram TF-IDF with class-balanced logistic regression. I retain `C=0.1`, `min_df=2`, sublinear term frequency, and the fixed random state. This checkpoint constructs and verifies the pipeline but does not fit it.

In [3]:
# I construct the frozen candidate exactly as preregistered for Week 6.
def make_frozen_candidate():
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    stop_words="english",
                    ngram_range=(1, 1),
                    min_df=2,
                    sublinear_tf=True,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=0.1,
                    class_weight="balanced",
                    max_iter=1_000,
                    random_state=SEED,
                ),
            ),
        ]
    )


frozen_candidate = make_frozen_candidate()
parameters = frozen_candidate.get_params()
expected_parameters = {
    "tfidf__lowercase": True,
    "tfidf__stop_words": "english",
    "tfidf__ngram_range": (1, 1),
    "tfidf__min_df": 2,
    "tfidf__sublinear_tf": True,
    "classifier__C": 0.1,
    "classifier__class_weight": "balanced",
    "classifier__max_iter": 1_000,
    "classifier__random_state": SEED,
}
for parameter, expected_value in expected_parameters.items():
    assert parameters[parameter] == expected_value, f"Unexpected value for {parameter}."

candidate_configuration = pd.DataFrame(
    [{"parameter": key, "frozen_value": repr(value)} for key, value in expected_parameters.items()]
)
display(candidate_configuration)
print("Frozen candidate configuration: verified; model not fitted in Checkpoint 1.")

,parameter,frozen_value
0,tfidf__lowercase,True
1,tfidf__stop_words,'english'
2,tfidf__ngram_range,"(1, 1)"
3,tfidf__min_df,2
4,tfidf__sublinear_tf,True
5,classifier__C,0.1
6,classifier__class_weight,'balanced'
7,classifier__max_iter,1000
8,classifier__random_state,42


Frozen candidate configuration: verified; model not fitted in Checkpoint 1.


## Data

The next cells validate the approved manifest, recover only the 199 reviewed source records from the local Enron file, and rebuild the same model text used in Week 4. The notebook displays aggregate summaries only.

### Validate the reviewed label manifest

In [4]:
# The manifest is the label source of truth. These checks stop missing, duplicate, or unapproved labels.
approved_labels = pd.read_csv(LABEL_MANIFEST)
required_label_columns = {
    "message_id",
    "sampling_stratum",
    "urgency_label",
    "label_version",
    "review_status",
}

assert set(approved_labels.columns) == required_label_columns
assert len(approved_labels) == EXPECTED_RECORDS
assert approved_labels["message_id"].is_unique
assert approved_labels[list(required_label_columns)].notna().all().all()
assert approved_labels["review_status"].eq("Reviewed").all()
assert approved_labels["label_version"].astype(str).eq("1.0").all()
assert set(approved_labels["urgency_label"]) == set(LABEL_ORDER)
assert approved_labels["urgency_label"].value_counts().to_dict() == EXPECTED_LABEL_COUNTS
assert approved_labels["sampling_stratum"].value_counts().to_dict() == {
    "general_random": 149,
    "urgency_enriched": 50,
}

manifest_summary = pd.DataFrame(
    {
        "records": approved_labels["urgency_label"].value_counts().reindex(LABEL_ORDER),
    }
).rename_axis("urgency_label")
display(manifest_summary)
display(
    pd.crosstab(
        approved_labels["sampling_stratum"],
        approved_labels["urgency_label"],
        margins=True,
    )
)
print("Reviewed label manifest: all validation checks passed.")

,records
urgency_label,
nonurgent,169
urgent,30


urgency_label,nonurgent,urgent,All
sampling_stratum,,,
general_random,131,18,149
urgency_enriched,38,12,50
All,169,30,199


Reviewed label manifest: all validation checks passed.


### Recover and validate the reviewed model text

The raw source contains about 1.4 GB of email data. I scan it in bounded chunks and retain only records listed in the approved manifest. The parser matches the Week 4 text contract by combining the subject and plain-text body for modeling. Raw text remains in memory only and is not displayed or exported.

In [5]:
# I recover only approved records and keep message content out of notebook outputs.
requested_ids = set(approved_labels["message_id"])
source_rows = []

for chunk in pd.read_csv(
    RAW_DATA,
    usecols=["file", "message"],
    chunksize=10_000,
):
    matched = chunk.loc[chunk["file"].isin(requested_ids), ["file", "message"]]
    if not matched.empty:
        source_rows.append(matched)

if not source_rows:
    raise RuntimeError("No reviewed source records were recovered.")

raw_modeling_rows = pd.concat(source_rows, ignore_index=True)
recovered_ids = set(raw_modeling_rows["file"])
assert len(raw_modeling_rows) == EXPECTED_RECORDS
assert raw_modeling_rows["file"].is_unique
assert recovered_ids == requested_ids
assert raw_modeling_rows["message"].notna().all()
assert raw_modeling_rows["message"].str.strip().ne("").all()


def parse_email(raw_message):
    parsed = Parser(policy=policy.default).parsestr(raw_message)
    subject = str(parsed.get("subject", "")).strip()
    try:
        if parsed.is_multipart():
            body_part = parsed.get_body(preferencelist=("plain",))
            body = body_part.get_content() if body_part else ""
        else:
            body = parsed.get_content()
    except Exception:
        body = str(parsed.get_payload())
    return f"{subject}\n\n{str(body).strip()}".strip()


raw_modeling_rows["combined_text"] = raw_modeling_rows["message"].map(parse_email)
modeling_data = (
    approved_labels[["message_id", "sampling_stratum", "urgency_label"]]
    .merge(
        raw_modeling_rows[["file", "combined_text"]],
        left_on="message_id",
        right_on="file",
        how="inner",
        validate="one_to_one",
    )
    .drop(columns="file")
)

assert len(modeling_data) == EXPECTED_RECORDS
assert modeling_data["message_id"].is_unique
assert modeling_data["combined_text"].notna().all()
assert modeling_data["combined_text"].str.strip().ne("").all()
assert modeling_data["urgency_label"].value_counts().to_dict() == EXPECTED_LABEL_COUNTS

# Remove the raw-message table after the approved model text is constructed.
del raw_modeling_rows, source_rows
gc.collect()

modeling_population_summary = pd.DataFrame(
    [
        {"check": "Reviewed source records recovered", "observed": len(modeling_data), "expected": EXPECTED_RECORDS},
        {"check": "Unique message identifiers", "observed": modeling_data["message_id"].nunique(), "expected": EXPECTED_RECORDS},
        {"check": "Nonempty model text", "observed": int(modeling_data["combined_text"].str.strip().ne("").sum()), "expected": EXPECTED_RECORDS},
        {"check": "Raw email text displayed or exported", "observed": 0, "expected": 0},
    ]
)
display(modeling_population_summary)
print("Reviewed modeling population: all validation checks passed.")

,check,observed,expected
0,Reviewed source records recovered,199,199
1,Unique message identifiers,199,199
2,Nonempty model text,199,199
3,Raw email text displayed or exported,0,0


Reviewed modeling population: all validation checks passed.


### Recreate and validate the preserved split

I recreate the Week 3 split with `random_state=42`, a 20% holdout, and label stratification. The training partition is the only population available for Week 6 model and threshold decisions. This cell validates membership and aggregate counts without displaying message identifiers or text.

In [6]:
# I preserve the prior 159/40 split and keep the holdout sealed for later preliminary comparison.
X_train, X_holdout, y_train, y_holdout = train_test_split(
    modeling_data["combined_text"],
    modeling_data["urgency_label"],
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=modeling_data["urgency_label"],
)

assert len(X_train) == EXPECTED_TRAIN_RECORDS
assert len(X_holdout) == EXPECTED_HOLDOUT_RECORDS
assert y_train.value_counts().to_dict() == EXPECTED_TRAIN_LABEL_COUNTS
assert y_holdout.value_counts().to_dict() == EXPECTED_HOLDOUT_LABEL_COUNTS
assert set(X_train.index).isdisjoint(set(X_holdout.index))
assert set(X_train.index) | set(X_holdout.index) == set(modeling_data.index)

split_summary = pd.DataFrame(
    {
        "training": y_train.value_counts().reindex(LABEL_ORDER),
        "preliminary_holdout": y_holdout.value_counts().reindex(LABEL_ORDER),
    }
).fillna(0).astype(int)
split_summary.loc["total"] = split_summary.sum()
display(split_summary)


def text_fingerprint(text):
    normalized = re.sub(r"\s+", " ", str(text).lower()).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


train_fingerprints = set(X_train.map(text_fingerprint))
holdout_fingerprints = set(X_holdout.map(text_fingerprint))
exact_cross_split_fingerprints = train_fingerprints & holdout_fingerprints
assert not exact_cross_split_fingerprints

split_validation = pd.DataFrame(
    [
        {"check": "Training records", "observed": len(X_train), "expected": EXPECTED_TRAIN_RECORDS},
        {"check": "Preliminary holdout records", "observed": len(X_holdout), "expected": EXPECTED_HOLDOUT_RECORDS},
        {"check": "Membership overlap", "observed": len(set(X_train.index) & set(X_holdout.index)), "expected": 0},
        {"check": "Exact normalized-text fingerprints shared across splits", "observed": len(exact_cross_split_fingerprints), "expected": 0},
    ]
)
display(split_validation)
print("Preserved split: all validation checks passed; holdout remains unused for selection.")

,training,preliminary_holdout
urgency_label,,
nonurgent,135,34
urgent,24,6
total,159,40


,check,observed,expected
0,Training records,159,159
1,Preliminary holdout records,40,40
2,Membership overlap,0,0
3,Exact normalized-text fingerprints shared acro...,0,0


Preserved split: all validation checks passed; holdout remains unused for selection.


## Results

Checkpoint 1 passed its required validations:

- The approved manifest contains 199 reviewed records with 169 nonurgent and 30 urgent labels.
- All 199 approved identifiers were recovered from the local source, and every record produced nonempty model text.
- The deterministic stratified split contains 159 training messages and 40 preliminary holdout messages.
- The training partition contains 135 nonurgent and 24 urgent messages; the holdout contains 34 nonurgent and 6 urgent messages.
- The two partitions have no membership overlap or exact normalized-text fingerprint overlap.
- The frozen Week 4 pipeline matches the preregistered configuration and remains unfitted at this checkpoint.

## Takeaways

The Week 6 experiment has a validated starting population, a reproducible split, and a frozen candidate. The next checkpoint will implement nested stratified cross-validation and threshold selection using only the 159-message training partition. The holdout will remain sealed until the method and final threshold are frozen.